# Université Paul Sabatier
## M1 IAFA — Foundations of Information Retrieval — 2026

Instructors: Lynda Tamine, Karim Radouane and Ahmed Rayane Kebir

Notebook proposed by : Jesús Lovón and Ahmed Rayane Kebir

---

> 💡 Develop reusable helper functions throughout this PW to keep your code clean and modular.

### Attention❗ About TP grading:
🚨 *Code questions*: Fill in the missing code in the corresponding sections (commented code gets the best marks).

🚨 *Open questions*: Write your textual answer as a comment in the corresponding cells.

🚨 *Keep your outputs*: **Empty outputs (notebook or non-executed cells) correspond to 0 points**.

---

# TP 5 — Retrieval-Augmented Generation (RAG) with LangChain

## Objectives
In this PW you will:
1. Load and preprocess a biomedical QA corpus.
2. Build a vector store using **FAISS** and **LangChain**.
3. Implement a complete **RAG pipeline** using LangChain's LCEL abstractions.
4. Evaluate using **ROUGE** scores.
5. **Experiment** with different LLMs, retrievers and prompt templates.

We use a mini version of the [BioASQ dataset](https://huggingface.co/datasets/enelpol/rag-mini-bioasq).

> 🖥️ This PW requires **GPU runtime**.

---
## Installations and Setup

In [ ]:
!pip install --quiet langchain langchain-community langchain-huggingface
!pip install --quiet faiss-cpu sentence-transformers
!pip install --quiet transformers accelerate bitsandbytes sentencepiece
!pip install --quiet datasets pandas evaluate rouge_score

---
## I. Data Loading and Preprocessing

We load the `enelpol/rag-mini-bioasq` dataset, which contains:
- A **text corpus** of biomedical passages.
- A **question-answer** split with questions and reference answers.

In [ ]:
from datasets import load_dataset

DATASET_NAME = "enelpol/rag-mini-bioasq"

question_answer_ds = load_dataset(DATASET_NAME, "question-answer-passages")
corpus_ds          = load_dataset(DATASET_NAME, "text-corpus")

print("Corpus splits    :", corpus_ds)
print("QA splits        :", question_answer_ds)
print("\nCorpus sample    :", corpus_ds["test"][0])
print("\nQA sample        :", question_answer_ds["test"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

question-answer-passages/train-00000-of-(…):   0%|          | 0.00/1.12M [00:00<?, ?B/s]

question-answer-passages/test-00000-of-0(…):   0%|          | 0.00/187k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4012 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/707 [00:00<?, ? examples/s]

text-corpus/test-00000-of-00001.parquet:   0%|          | 0.00/35.3M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/40181 [00:00<?, ? examples/s]

Corpus splits    : DatasetDict({
    test: Dataset({
        features: ['passage', 'id'],
        num_rows: 40181
    })
})
QA splits        : DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'id', 'relevant_passage_ids'],
        num_rows: 4012
    })
    test: Dataset({
        features: ['question', 'answer', 'id', 'relevant_passage_ids'],
        num_rows: 707
    })
})

Corpus sample    : {'passage': 'New data on viruses isolated from patients with subacute thyroiditis de Quervain \nare reported. Characteristic morphological, cytological, some physico-chemical \nand biological features of the isolated viruses are described. A possible role \nof these viruses in human and animal health disorders is discussed. The isolated \nviruses remain unclassified so far.', 'id': 9797}

QA sample        : {'question': 'Is capmatinib effective for glioblastoma?', 'answer': 'No. Combination of capmatinib buparlisib resulted in no clear activity in patients with recurren

### ✍️ Question 1 OK — Data Extraction

Create three lists:
- `corpus_texts` — all document strings from the corpus.
- `questions_test` — questions from the test split.
- `golden_answers` — corresponding reference answers from the test split.

In [ ]:
corpus_texts   = corpus_ds["test"]["passage"]

questions_test = question_answer_ds["test"]["question"]
golden_answers = question_answer_ds["test"]["answer"]

print(f"Corpus      : {len(corpus_texts)} documents")
print(f"Questions   : {len(questions_test)}")
print(f"\nSample doc  : {corpus_texts[0][:200]}...")
print(f"\nSample Q    : {questions_test[0]}")
print(f"Sample A    : {golden_answers[0]}")

Corpus      : 40181 documents
Questions   : 707

Sample doc  : New data on viruses isolated from patients with subacute thyroiditis de Quervain 
are reported. Characteristic morphological, cytological, some physico-chemical 
and biological features of the isolate...

Sample Q    : Is capmatinib effective for glioblastoma?
Sample A    : No. Combination of capmatinib buparlisib resulted in no clear activity in patients with recurrent PTEN-deficient glioblastoma.


---
## II. Building a LangChain Vector Store (FAISS)

LangChain wraps FAISS into a `FAISS` vector store paired with any `Embeddings` object.
The key abstraction is:

```
Documents  →  Embeddings model  →  FAISS vector store  →  Retriever
```

We use `HuggingFaceEmbeddings` wrapping `all-MiniLM-L6-v2` — a fast, lightweight model
that produces 384-dimensional sentence embeddings.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Wrap raw strings as LangChain Document objects
documents = [Document(page_content=t) for t in corpus_texts]

# Load the embedding model (downloads ~90MB on first run)
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True}
)

# Build FAISS index — this encodes all documents
vectorstore = FAISS.from_documents(documents, embeddings)

print(f"Vector store built — {vectorstore.index.ntotal} vectors, dim={vectorstore.index.d}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store built — 40181 vectors, dim=384


### 💬 Understanding Check OK — Embeddings and Vector Search

Answer the following questions **as comments** in the cell below.

**a)** `all-MiniLM-L6-v2` produces 384-dimensional vectors. What does each dimension represent, and why is it not directly interpretable like a TF-IDF feature?

**b)** When we call `FAISS.from_documents(documents, embeddings)`, the corpus is encoded **once** and stored. At query time, only the query is encoded. Why is this asymmetry important for the scalability of a retrieval system? What would be the cost if we re-encoded all documents at every query?

In [ ]:
## ANSWER a)
# Each dimensions represents a semantic feature learned by the model during the training on the text corpus.
# In general this type of vectors allows the model to capture the context and meaning of a sentence rather than just looking for exact character matches.
# The meaning is spread across the vector rather than toindividual vocabulary terms.

## ANSWER b)
# This asimmetry is caused by the fact that normally the corpus is huge, while the query  is a single sentence
# It allows the most intensive task (encoding the huge corpus) to be perfomed only once offline.
# And when the code is ran again the system will only need to encode the single incoming query.
# If we were forced to re-encode the entire corpus for every query, the system would suffer from high lattency and high GPU costs, turning every search into a processthat could take minutes.

### ✍️ Question 2 OK — Retriever

Create a LangChain retriever from the vector store that returns the top-`k=2` most relevant documents.
Then retrieve documents for the sample query below and print them.

In [ ]:
SAMPLE_QUERY = "Neurostimulation of which nucleus is used for treatment of dystonia?"
TOP_K = 2

# Create a retriever from the vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# Retrieve documents for the sample query
retrieved_docs = retriever.invoke(SAMPLE_QUERY)

print(f"Query: {SAMPLE_QUERY}\n")
for i, doc in enumerate(retrieved_docs):
    print(f"[Doc {i}] {doc.page_content[:300]}\n{'—'*60}")

Query: Neurostimulation of which nucleus is used for treatment of dystonia?

[Doc 0] We report on the effects of bilateral neurostimulation of the ventral 
intermediate thalamic nucleus (VIM) in a patient with medically intractable and 
progressing inherited myoclonus dystonia syndrome (IMDS). Postoperatively, the 
patient improved by approximately 80% on the modified version of a m
————————————————————————————————————————————————————————————
[Doc 1] A 70 year old woman presented with a 6 year history of medically refractory 
severe tardive dystonia. After informed consent, a bilateral stereotactic 
electrode placement targeting the ventral intermediate thalamic nucleus (VIM) 
and the globus pallidus internus (GPi) was performed. After bilateral
————————————————————————————————————————————————————————————


---
## III. LLM Setup (Reader)

We load **Falcon3-1B-Base** via a HuggingFace pipeline, then wrap it with LangChain's
`HuggingFacePipeline` so it slots cleanly into any LangChain chain.

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_huggingface import HuggingFacePipeline

MODEL_ID = "tiiuae/Falcon3-1B-Base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

hf_pipeline = transformers.pipeline(
    "text-generation",
    model=MODEL_ID,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
    max_new_tokens=100,
    do_sample=True,
    top_k=10,
    eos_token_id=tokenizer.eos_token_id,
    return_full_text=False,   # return only the generated part, not the prompt
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("LLM loaded:", MODEL_ID)

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/91.0 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_k', 'eos_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded: tiiuae/Falcon3-1B-Base


---
## IV. Building the RAG Chain with LangChain LCEL

LangChain's **Expression Language (LCEL)** composes chains with the `|` operator:

```
retriever  →  format_docs  →  prompt  →  llm  →  output_parser
```

Each component is a `Runnable`, making the pipeline easy to inspect, debug, and swap.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Helper: merge a list of Documents into one context string ──────────────
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── Helper: build any RAG chain from its 3 components ─────────────────────
def build_rag_chain(retriever, prompt, llm):
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

# ── Baseline prompt ───────────────────────────────────────────────────────
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are given a QUESTION and a CONTEXT.\n"
        "Provide only the concise ANSWER contained in the context.\n\n"
        "CONTEXT:\n{context}\n\n"
        "QUESTION: {question}\n\n"
        "ANSWER:"
    )
)

rag_chain = build_rag_chain(retriever, RAG_PROMPT, llm)

# ── Quick sanity check ────────────────────────────────────────────────────
answer = rag_chain.invoke(SAMPLE_QUERY)
print("Query  :", SAMPLE_QUERY)
print("Answer :", answer.strip())

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query  : Neurostimulation of which nucleus is used for treatment of dystonia?
Answer : Neurostimulation of the ventral intermediate thalamic nucleus (VIM) is used for treatment of dystonia.

Explanation:
The question asks for the nucleus that is used for treatment of dystonia. The context provides information about the effectiveness of bilateral neurostimulation of the ventral intermediate thalamic nucleus (VIM) in a patient with medically intractable and progressing inherited myoclonus dystonia syndrome (IMDS). The patient improved by approximately 80% on the modified version of


### ✍️ Question 3 OK — Batch Evaluation Function

Implement `evaluate_rag(questions, chain)` that takes a list of question strings and a
LangChain chain and returns a list of generated answer strings.

Then run it on the first 10 test questions and compute the baseline ROUGE scores.

In [ ]:
import evaluate as hf_evaluate

rouge_metric = hf_evaluate.load("rouge")

def evaluate_rag(questions, chain):
    """
    Run the RAG chain on every question and return the list of generated answers.

    Parameters
    ----------
    questions : list[str]
    chain     : a LangChain Runnable

    Returns
    -------
    list[str]
    """
    answers = []

    # Your code here
    # Iterate through each question and invoke the chain
    for q in questions:
        raw_answer = chain.invoke(q)
        # Clean up the answer: strip whitespace and take only the first line
        answers.append(raw_answer.strip().split("\n")[0])
    return answers


def compute_rouge(predictions, references, label=""):
    """Compute and pretty-print ROUGE scores. Returns the result dict."""
    result = rouge_metric.compute(predictions=predictions, references=references)
    header = f"=== {label} ===" if label else "=== ROUGE ==="
    print(header)
    for k, v in result.items():
        print(f"  {k}: {v:.4f}")
    return result

# Evaluate baseline
N_EVAL = 10
predicted_baseline = evaluate_rag(questions_test[:N_EVAL], rag_chain)
rouge_baseline = compute_rouge(predicted_baseline, golden_answers[:N_EVAL],
                               label="Baseline: MiniLM + FlatL2 + Falcon3")


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

=== Baseline: MiniLM + FlatL2 + Falcon3 ===
  rouge1: 0.2533
  rouge2: 0.1226
  rougeL: 0.2167
  rougeLsum: 0.2157


### 💬 Analysis OK — Interpreting Your Baseline ROUGE Scores

Look at the ROUGE-1, ROUGE-2, and ROUGE-L scores you just obtained and answer as comments below.

**a)** In your own words, what does ROUGE-1 measure that ROUGE-2 does not? Given that BioASQ answers are short factoid phrases (e.g., "globus pallidus internus"), which metric do you expect to be most discriminative, and why?

In [ ]:
## Your answer here (write as comments)
# Rouge-1 measures the overlap of individual words between the generated and reference answers.
# It captures the presence of correct keywords regardless of their sequence.

# Rouge-2 measures the overlap of consecutive word pairs (like bigrams), whiwh captures the local word order.

# Personnaly I think Rouge-2 might be more usefull in medical context because it will be a more discrimintaive metric
# In medical documents I think there are many multi-word terms like "globus pallidus internus" for which Rouge-2 is ideal, it will reward only if the exact sequence of words is found.
# Whereas Rouge-1 might reward the model for simply finding the right words no mather the order.

print("Sample predictions vs. golden answers:\n")
for i in range(min(5, len(predicted_baseline))):
    print(f"Q : {questions_test[i]}")
    print(f"Pred   : {predicted_baseline[i]}")
    print(f"Golden : {golden_answers[i]}")
    print()
print("ROUGE interpretation answered.")

Sample predictions vs. golden answers:

Q : Is capmatinib effective for glioblastoma?
Pred   : No. The study found that the addition of capmatinib to Temozolomide plus radiotherapy (TEM+RAD) did not improve the OS of patients with GBM; however, it did improve PFS in patients treated by cilengitide who could not get improvement in OS. The rate of adverse effects higher than grade 3 was 57.7% in the experimental group and 44.1% in the placebo group (odds ratio, 1.6
Golden : No. Combination of capmatinib buparlisib resulted in no clear activity in patients with recurrent PTEN-deficient glioblastoma.

Q : Describe the mechanism of action of ibalizumab.
Pred   : Ibalizumab is a humanized monoclonal antibody that binds human CD4, the 
Golden : Ibalizumab is a humanized monoclonal antibody that acts as post-attachment inhibitor by binding CD4 2nd domain of T lymphocyte and preventing HIV connection to CCR5 or CXCR4. It has been recently approved by Food and Drug Administration as a new intrav

---
## V. Experiment 1 -  Quantizing Falcon3 — Memory vs. Quality

The baseline loads Falcon3-1B-Base in **bfloat16**, which halves memory vs. float32
but keeps full 16-bit precision per weight.

In this question you will reload Falcon3 with **8-bit** and **4-bit NF4** quantization,
measure the GPU memory footprint of each variant, run the full RAG evaluation pipeline,
and compare ROUGE scores across all three precisions.

1. Reload Falcon3 with **8-bit** quantization (`load_in_8bit=True`).
2. Reload Falcon3 with **4-bit NF4** quantization (`load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`).
3. For each variant: print GPU memory, build a RAG chain, run `evaluate_rag`, compute ROUGE.
4. Answer the conceptual questions in the analysis cell below.

### ✍️ Question 4 OK : Quantization 8bit and 4bit


In [ ]:
import gc
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_huggingface import HuggingFacePipeline

# ── Setup ────────────────────────────────────────────────────────────────

def gpu_mem(label):
    torch.cuda.synchronize()
    mem      = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved()  / 1024**3
    print(f"[{label}] allocated: {mem:.2f} GB  |  reserved: {reserved:.2f} GB")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

# ── 1. BASELINE (bfloat16) ───────────────────────────────────────────────
gpu_mem("Falcon3 bfloat16")
rag_chain = build_rag_chain(retriever, RAG_PROMPT, llm)
predicted = evaluate_rag(questions_test[:N_EVAL], rag_chain)
rouge_base = compute_rouge(predicted, golden_answers[:N_EVAL], label="Baseline")

# DELETE BASELINE BEFORE NEXT STEP
del hf_pipeline, llm, rag_chain
cleanup()

# ── 2. 8-BIT QUANTIZATION ────────────────────────────────────────────────
quant_config_8bit = transformers.BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    return_full_text=False ######### rajout
)
model_8bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant_config_8bit, device_map="auto", trust_remote_code=True
)
pipe_8bit = transformers.pipeline(
    "text-generation", model=model_8bit, tokenizer=tokenizer, max_new_tokens=100
)
llm_8bit = HuggingFacePipeline(pipeline=pipe_8bit)

gpu_mem("Falcon3 8-bit")
rag_chain_8bit = build_rag_chain(retriever, RAG_PROMPT, llm_8bit)
predicted_8bit = evaluate_rag(questions_test[:N_EVAL], rag_chain_8bit)
rouge_8bit     = compute_rouge(predicted_8bit, golden_answers[:N_EVAL],
                               label="Falcon3 8-bit")

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Falcon3 bfloat16] allocated: 3.21 GB  |  reserved: 3.52 GB


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

=== Baseline ===
  rouge1: 0.2819
  rouge2: 0.0860
  rougeL: 0.2296
  rougeLsum: 0.2341


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[Falcon3 8-bit] allocated: 2.16 GB  |  reserved: 3.16 GB


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

=== Falcon3 8-bit ===
  rouge1: 0.0500
  rouge2: 0.0000
  rougeL: 0.0499
  rougeLsum: 0.0501


In [ ]:
## Your Code
# DELETE 8-BIT BEFORE NEXT STEP
del quant_config_8bit, model_8bit, pipe_8bit, llm_8bit, rag_chain_8bit, rouge_8bit
cleanup()

# ── 3. 4-BIT NF4 QUANTIZATION ────────────────────────────────────────────

## Your code Here
quant_config_4bit = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bf16 for speed/stability
    bnb_4bit_use_double_quant=True,      # Quantize weights and quantization constants
    return_full_text=False ############### rajout
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant_config_4bit, device_map="auto", trust_remote_code=True
)
pipe_4bit = transformers.pipeline(
    "text-generation", model=model_4bit, tokenizer=tokenizer, max_new_tokens=100
)
llm_4bit = HuggingFacePipeline(pipeline=pipe_4bit)

gpu_mem("Falcon3 4-bit NF4")
rag_chain_4bit = build_rag_chain(retriever, RAG_PROMPT, llm_4bit)
predicted_4bit = evaluate_rag(questions_test[:N_EVAL], rag_chain_4bit)
rouge_4bit     = compute_rouge(predicted_4bit, golden_answers[:N_EVAL],
                               label="Falcon3 4-bit NF4")

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Falcon3 4-bit NF4] allocated: 1.65 GB  |  reserved: 2.93 GB


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

=== Falcon3 4-bit NF4 ===
  rouge1: 0.0500
  rouge2: 0.0000
  rougeL: 0.0499
  rougeLsum: 0.0501


In [ ]:
print("--- 8-bit Sample Prediction ---")
print(predicted_8bit[0])

print("\n--- 4-bit Sample Prediction ---")
print(predicted_4bit[0])

--- 8-bit Sample Prediction ---
You are given a QUESTION and a CONTEXT.

--- 4-bit Sample Prediction ---
You are given a QUESTION and a CONTEXT.


In [ ]:
## Comments on results

# I observed that reducing the precision to 8-bit and 4-bit significantly lowered GPU memory usage from 4.76 GB down to 3.19 GB, but the model's actual performance completely broke down.
# When I searched why there are identical ROUGE-1 scores and the zero ROUGE-2 values I found that it happens because of a phenomenon called mode collapse.
# Basically, the Falcon3-1B model is too small to handle such high compression, which causes its internal attention mechanism to fail.
# Instead of processing the medical context to answer the question, the model just gets stuck and repeats the first line of the prompt instructions.
# This shows that while quantization is helpful for saving RAM, very small models are often too fragile to keep their intelligence once you start cutting down their bit-width.

### 💬 Analysis OK — Quantization Trade-offs

Answer as comments in the cell below.

**a)** Fill in the table from your measurements:

| Precision | Bits/weight | Theoretical size (1B params) | Measured GPU mem | ROUGE-1 |
|-----------|-------------|------------------------------|------------------|---------|
| bfloat16  | 16          | ~2 GB                        |     4.7 GB             |     0.2849    |
| int8      | 8           | ~1 GB                        |    3.7 GB              |   0.0508      |
| NF4       | 4           | ~0.5 GB                      |         3.1 GB         |    0.0508     |

**b)** bfloat16 and float16 both use 16 bits but have different exponent/mantissa splits.
Why is **bfloat16** preferred over float16 for LLM inference on modern GPUs?

**c)** The measured GPU memory does not decrease exactly by a factor of 2 between
each precision level. What else contributes to GPU memory usage beyond the model weights?

In [ ]:
### Your answer here
## ANSWER b)
# bfloat16 is preferred over float16 because it shares the same dynamic range as float32.
# float16 has more precision, it has a very large exponent range, which can lead to a numerical overflow.
# Using bfloat16 ensures the model doesn't crash during inference, making it much more stable.

## ANSWER c)
# Things that can contribute to GPU memory usage beyond the model weights are:
# - the embedding model "all-MiniLM-L6-v2" is also loaded in the GPU memory to convert text into vectors.
# - temporary tensors can be created while the mode calculates the relationship between the question and the context.
# - software drivers and basic mathematical kernels required to run on the GPU.
# - cache that stores the history of the conversation grows with the mength of the context.


---
## VI. Experiment 2 — Alternative LLM (DeepSeek-R1, 4-bit)

We swap Falcon3 for a **quantized DeepSeek-R1-Distill-Qwen-1.5B** model.
4-bit quantization (via `bitsandbytes` NF4) reduces memory by ~8× with minimal
quality loss.

### ✍️ Question 5 OK — Swap the Reader
Load a different LLM, plug it into your best-performing chain, evaluate, and compare.

In [ ]:
MODEL_NEW = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

tokenizer_new = AutoTokenizer.from_pretrained(MODEL_NEW)

# 4-bit quantization — maximises memory efficiency
quantization_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"       # NormalFloat4 — best quality for LLMs
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NEW,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

hf_pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer_new,
    trust_remote_code=True,
    device_map="auto",
    max_new_tokens=100,
    do_sample=True,
    top_k=10,
    eos_token_id=tokenizer_new.eos_token_id,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("Loaded:", MODEL_NEW)

#### Your code here
rag_chain_new_llm = build_rag_chain(retriever, RAG_PROMPT, llm)
predicted_new_llm = evaluate_rag(questions_test[:N_EVAL], rag_chain_new_llm)

rouge_new_llm = compute_rouge(predicted_new_llm, golden_answers[:N_EVAL],
                              label=f"Exp 6: {MODEL_NEW.split('/')[1]} (4-bit)")

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loaded: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

=== Exp 6: DeepSeek-R1-Distill-Qwen-1.5B (4-bit) ===
  rouge1: 0.1478
  rouge2: 0.0274
  rougeL: 0.1203
  rougeLsum: 0.1160


In [ ]:
### Comments
# These results are a major improvement over the Falcon3 experiment.
# Unlike the previous 4-bit and 8-bit runs that completely collapsed, the DeepSeek distilled model is actually functional, as shown by the non-zero ROUGE-2 score (0.0257).
# While its ROUGE-1 (0.1362) is still lower than the original bfloat16 baseline (~0.28), it proves that this model is far more resilient to 4-bit quantization.
# I think the reason why DeepSeek performs better even at 4-bit is due to Knowledge Distillation.
# This model was specifically trained to mimic the reasoning patterns of a much larger parent model (full DeepSeek-R1), giving it a more stable internal logic, that survives compression better than the base Falcon3 architecture.

---
## VII. Experiment 3 — Better Retriever (DRAGON+)

`facebook/dragon-plus-context-encoder` is a **dual-encoder** retriever trained with
hard-negative mining on a large set of open-domain QA pairs. Its query encoder
(`dragon-plus-query-encoder`) produces query embeddings that are aligned with the
context encoder space — leading to better retrieval than general-purpose encoders.

### ✍️ Question 6 OK — DRAGON+ Retriever
1. Build a new vector store using the DRAGON+ context encoder.
2. Rebuild the RAG chain and evaluate.
3. Comment on the difference vs MiniLM.

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings
from typing import List

# DRAGON uses separate encoders for queries and documents.
# Subclass LangChain's Embeddings to use the right model for each direction.
# Context encoder : "facebook/dragon-plus-context-encoder"
# Query encoder   : "facebook/dragon-plus-query-encoder"

class DragonEmbeddings(Embeddings):
    def __init__(self):
        #### Your code here
        self.query_encoder = SentenceTransformer("facebook/dragon-plus-query-encoder")
        self.context_encoder = SentenceTransformer("facebook/dragon-plus-context-encoder")


    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Called when indexing corpus passages."""
        #### Your code here
        return self.context_encoder.encode(texts, show_progress_bar=True).tolist()

    def embed_query(self, text: str) -> List[float]:
        """Called when encoding an incoming query."""
        #### Your code here
        return self.query_encoder.encode(text).tolist()

print("Loading DRAGON+ encoders...")
embeddings_dragon = DragonEmbeddings()

print("Building DRAGON+ vector store...")
#### Your code here
vectorstore_dragon = FAISS.from_documents(documents, embeddings_dragon)
retriever_dragon   = vectorstore_dragon.as_retriever(search_kwargs={"k": 2})

rag_chain_dragon = build_rag_chain(retriever_dragon, RAG_PROMPT, llm)
predicted_dragon = evaluate_rag(questions_test[:N_EVAL], rag_chain_dragon)
rouge_dragon = compute_rouge(predicted_dragon, golden_answers[:N_EVAL],
                             label="Exp 2: DRAGON+ Retriever")


Loading DRAGON+ encoders...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: facebook/dragon-plus-query-encoder
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Building DRAGON+ vector store...


Batches:   0%|          | 0/1256 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

=== Exp 2: DRAGON+ Retriever ===
  rouge1: 0.1197
  rouge2: 0.0161
  rougeL: 0.0946
  rougeLsum: 0.0965


In [ ]:
## Comments
# In general the difference between these two retrievers is their architecture.
# MiniLM is a bi-encoder that uses single set weights to encode both questions ans documents.
# On the other side, DRAGON+ is a dual-encoder that uses two specialized models.
# The first one is dragon-plus-query-encoder used for short queries
# The second one is dragon-plus-context-encoder used for long context documents
# Normally I would have said that the dual-encoder setup is better at handling for example an asymetry of search (when a short question must find a match in a long passage).
# However the results are different, both ROUGE-1 scores: MiniLM (0.1362) and DRAGON+ (0.1197)
# As we can see DRAGON+ performed slightly worse. I think maybe the general purpose miniLM embeddings were already quite effective.
# My hipothesis is that a more complex model isn't always better.
# Instead, the performance depends more on how well the retriever's training data aligns with the specific vocabulary of the medical corpus that we are using.

### 💬 Analysis OK — Retriever Comparison

Answer as comments in the cell below.

**a)** The `DragonEmbeddings` class uses **two different models**: one for documents (`dragon-plus-context-encoder`) and one for queries (`dragon-plus-query-encoder`). Why are they different? What would happen to retrieval quality if you used the context encoder for both queries and documents?

In [ ]:
## Answer Here
# The two encoders are different because queries and documents have different characteristics.
# Queries generally are short, informal and epresent a search intent, while docuents are long, detailed and factual.
# A dual-encoder architecture uses specialized models to try solve this gap, that's why the short question and the long answer are mapped in a space where they can be accurately compared.
# If we use the context encoder for both queries and documents the quality would drop.
# The context encoder is trained to process dense, structured passages. If we give him a short query, it would produce a representation that doesn't align with the document embeddings.
# I think this could cause the retriever to mis relevant documents.

---
## VIII. Experiment 4 — Prompt Engineering

The **prompt template** heavily influences what the LLM extracts from the context.
We compare three styles:
- **Baseline** — simple instruction.
- **Prompt A** — strict one-sentence constraint + "only if found in context".
- **Prompt B** — chain-of-thought style asking the model to reason step-by-step.

### ✍️ Question 6 OK — Custom Prompts
Design One alternative prompt, evaluate it, and compare with the baseline.

In [ ]:
# ── Prompt A: strict, grounded, one-sentence ──────────────────────────────
PROMPT_A = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Answer the question using ONLY information from the context below.\n"
        "If the answer is not in the context, reply: 'Not found.'\n"
        "Give a single concise sentence. Do not repeat the question.\n\n"
        "Context: {context}\n\n"
        "Question: {question}\n"
        "Answer in one sentence:"
    )
)

# ── Prompt B: chain-of-thought ────────────────────────────────────────────
PROMPT_B = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are a biomedical expert. Read the context carefully.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Step 1 — Identify the key entities in the question.\n"
        "Step 2 — Locate the relevant sentence(s) in the context.\n"
        "Step 3 — State the final answer concisely.\n\n"
        "Final answer:"
    )
)

chain_prompt_a = build_rag_chain(retriever, PROMPT_A, llm)
chain_prompt_b = build_rag_chain(retriever, PROMPT_B, llm)

predicted_a = evaluate_rag(questions_test[:N_EVAL], chain_prompt_a)
predicted_b = evaluate_rag(questions_test[:N_EVAL], chain_prompt_b)

rouge_a = compute_rouge(predicted_a, golden_answers[:N_EVAL], label="Exp 3a: Prompt A (strict)")
rouge_b = compute_rouge(predicted_b, golden_answers[:N_EVAL], label="Exp 3b: Prompt B (CoT)")

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

=== Exp 3a: Prompt A (strict) ===
  rouge1: 0.0806
  rouge2: 0.0098
  rougeL: 0.0624
  rougeLsum: 0.0616
=== Exp 3b: Prompt B (CoT) ===
  rouge1: 0.0424
  rouge2: 0.0081
  rougeL: 0.0333
  rougeLsum: 0.0312


In [ ]:
# Design an alternative prompt template and test it.
# Ideas: strict one-sentence, chain-of-thought, few-shot, domain-specific instruction...

#### Your code here — define PROMPT_C

PROMPT_C = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are a highly precise biomedical assistant. Use the following context to provide "
        "a factual answer to the question. If the information is not in the context, "
        "clearly state that you cannot find the answer.\\n\\n"
        "Context:\\n{context}\\n\\n"
        "Question: {question}\\n\\n"
        "Expert Answer:"
    )
)

chain_prompt_c = build_rag_chain(retriever, PROMPT_C, llm)

predicted_c = evaluate_rag(questions_test[:N_EVAL], chain_prompt_c)

rouge_c = compute_rouge(predicted_c, golden_answers[:N_EVAL], label="Exp 3c: Prompt C")


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

=== Exp 3c: Prompt C ===
  rouge1: 0.2264
  rouge2: 0.0623
  rougeL: 0.1661
  rougeLsum: 0.1645


In [ ]:
## Answer
# Compared to the baseline, prompt C performed very well and closed a lot the gap that we saw earlier.
# Comparing the ROUGE-1 metric, while bfloat16 baseline had 0.2819 and 4bit deepseek with dragon+ got 0.1197, by simply changing the prompt ir pushed all the way up to 0.2264.
# I tried to give the prompt a expert personallity and it gave a huge improvement.
# So even if quantization hurted the model's performance, a better prompt can help recover and use the retrieved medical context more effectively.
# I also noticed that the ROUGE-2 score jumped from 0.016 to 0.062 which means the model is getting the specific medical word pairs correct, closer to the baseline's 0.086.

---
## X. Summary and Open Questions

### ✍️ Question 10 OK — Comparative Analysis

In [ ]:
import pandas as pd

# ── Collect all results into a summary table ──────────────────────────────
results = {
    "Baseline (FlatL2, MiniLM, default prompt, Falcon3)" : rouge_baseline,
    "Exp 1: DeepSeek-R1-Distill 4-bit"                  : rouge_new_llm,
    "Exp 2: DRAGON+ retriever"                           : rouge_dragon,
    "Exp 3a: Prompt A (strict)"                          : rouge_a,
    "Exp 3b: Prompt B (CoT)"                             : rouge_b,
    "Exp 3c: Prompt C "                             : rouge_c,
}

rows = []
for name, r in results.items():
    rows.append({
        "System"   : name,
        "ROUGE-1"  : round(r["rouge1"], 4),
        "ROUGE-2"  : round(r["rouge2"], 4),
        "ROUGE-L"  : round(r["rougeL"], 4),
    })

df = pd.DataFrame(rows).set_index("System")
df["ROUGE-1 Δ"] = (df["ROUGE-1"] - df["ROUGE-1"].iloc[0]).round(4)

print(df.to_string())

                                                    ROUGE-1  ROUGE-2  ROUGE-L  ROUGE-1 Δ
System                                                                                  
Baseline (FlatL2, MiniLM, default prompt, Falcon3)   0.2533   0.1226   0.2167     0.0000
Exp 1: DeepSeek-R1-Distill 4-bit                     0.1478   0.0274   0.1203    -0.1055
Exp 2: DRAGON+ retriever                             0.1197   0.0161   0.0946    -0.1336
Exp 3a: Prompt A (strict)                            0.0806   0.0098   0.0624    -0.1727
Exp 3b: Prompt B (CoT)                               0.0424   0.0081   0.0333    -0.2109
Exp 3c: Prompt C                                     0.2264   0.0623   0.1661    -0.0269


### 💬 Analysis — Final Comparative Analysis

Use the table printed above to answer the following questions as comments in the cells below.

**Question 10a** — Look at the ROUGE-1 Δ column. Rank the five modifications (LLM, retriever, prompt A, prompt B, prompt C) from most to least impactful. Does this ranking match your intuition? Explain *why* the top-ranked modification has the greatest effect by reasoning about the RAG pipeline architecture.


**Question 10b** — Fine-tuning strategy: describe a targeted fine-tuning approach that could further improve RAG performance beyond what you achieved by swapping components. Which component would you fine-tune, what training data would you use, what loss function, and why? *(Hint: review TP1.)*

In [ ]:
# Your answer here (write as comments)
# Looking at the values in the ROUGE-1 Δ column, the ranking from most to least impactful is Prompt B, Prompt A, DRAGON+ retriever, DeepSeek 4-bit and finally Prompt C.
# I found this pretty surprising because I thought that changing the entire retriever model or the LLM itself would be the biggest deal.
# Instead, the results show that the prompt instruction actually had the most power to move the scores.
# I think the Prompt B was the most impactful because it changes how the reader componentn in the rag pipeline behaves.
# By asking a chain of thought reasoning, it forces the LLM to generate a long explanation rthaer than a short answer (because it needs to prove that i followed each step).
# In a rag architecture, the reader is at the final stage that shapes the output, and if we change the instructions, it has a massiv effect on the ovrlap metrcis like ROUGE, even more than the quality of the documents retrieved.

print("Q10a answered.")

Q10a answered.


In [ ]:
# Your answer here (write as comments)
# To really boost the RAG performance, I would focus on fine-tuning the Retriever (the embedding model) rather than the LLM.
# If the retriever doesn't find the exact right context, the LLM is basically guessing, so that's why I think fxing the research would be more effective.
# I would use the same dataset mini-BioASQ, by using the same domain specific medical questions and documents, the model can learn the specific terminology (instead of doing a general model).
# As used in the first TP I would use a Triplet Loss, I think by taking a question, its correct document and incorrect document the question's vector are going to approach the good documents ones.
# It's going to match specialized medical questions to their specific answers, making the RAG pipileine much more reliable.

print("Q10b answered.")

Q10b answered.
